# Notebook 04 — Spatial Analysis

**PM Accelerator | Global Weather Forecasting**

This notebook creates interactive and static geographic visualizations:
- Folium temperature map (interactive)
- Folium anomaly map (interactive)
- Plotly scatter geo (interactive)
- GeoPandas static map
- Regional comparisons

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')

## 1. Load Clean Dataset

In [ ]:
from config import PROCESSED_DATA_DIR, TARGET_VARIABLE, LAT_COLUMN, LON_COLUMN, LOCATION_COLUMN

df_clean = pd.read_parquet(PROCESSED_DATA_DIR / 'weather_clean.parquet')
df_clean['last_updated'] = pd.to_datetime(df_clean['last_updated'], errors='coerce')
print('Clean dataset shape:', df_clean.shape)
df_clean[[LOCATION_COLUMN, LAT_COLUMN, LON_COLUMN, TARGET_VARIABLE]].head()

## 2. Aggregate by Location

In [ ]:
from spatial_analysis import aggregate_by_location, get_regional_stats

location_df = aggregate_by_location(df_clean)
regional_df = get_regional_stats(df_clean)

print('Unique locations:', len(location_df))
print('Unique countries:', len(regional_df))
location_df.head()

## 3. Plotly Scatter Geo — Temperature

In [ ]:
import plotly.express as px

df_plot = location_df.dropna(subset=[LAT_COLUMN, LON_COLUMN, TARGET_VARIABLE])

fig = px.scatter_geo(
    df_plot,
    lat=LAT_COLUMN,
    lon=LON_COLUMN,
    color=TARGET_VARIABLE,
    hover_name=LOCATION_COLUMN if LOCATION_COLUMN in df_plot.columns else None,
    color_continuous_scale='RdBu_r',
    title='Global Mean Temperature by Location',
    labels={TARGET_VARIABLE: 'Temp (°C)'},
    projection='natural earth',
)
fig.update_geos(showland=True, landcolor='lightgray', showocean=True, oceancolor='lightblue')
fig.update_layout(height=550, margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
fig.show()

## 4. Plotly Scatter Geo — Precipitation

In [ ]:
precip_col = 'precip_mm'
if precip_col in location_df.columns:
    df_precip = location_df.dropna(subset=[LAT_COLUMN, LON_COLUMN, precip_col])

    fig2 = px.scatter_geo(
        df_precip,
        lat=LAT_COLUMN,
        lon=LON_COLUMN,
        color=precip_col,
        size=precip_col,
        hover_name=LOCATION_COLUMN if LOCATION_COLUMN in df_precip.columns else None,
        color_continuous_scale='Blues',
        title='Global Mean Precipitation by Location',
        labels={precip_col: 'Precip (mm)'},
        projection='natural earth',
    )
    fig2.update_geos(showland=True, landcolor='lightyellow')
    fig2.update_layout(height=550)
    fig2.show()

## 5. Folium Interactive Map

In [ ]:
from spatial_analysis import create_temperature_map
from IPython.display import IFrame

map_path = create_temperature_map(location_df)
if map_path:
    IFrame(str(map_path), width='100%', height='500')

## 6. Regional Summary

In [ ]:
top20 = regional_df.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdBu_r(np.linspace(0, 1, len(top20)))
ax.barh(top20['country'][::-1], top20['mean_temp'][::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Top 20 Countries by Mean Temperature', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Temperature (°C)')
plt.tight_layout()
plt.show()

## 7. Temperature Variability by Region

In [ ]:
# Scatter: mean vs std dev across countries
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    regional_df['mean_temp'],
    regional_df['std_temp'],
    alpha=0.7,
    s=regional_df['count'] / regional_df['count'].max() * 200 + 10,
    c=regional_df['mean_temp'],
    cmap='RdBu_r',
    edgecolors='gray',
    linewidths=0.5,
)

for _, row in regional_df.head(10).iterrows():
    ax.annotate(row['country'], (row['mean_temp'], row['std_temp']),
                fontsize=7, alpha=0.8)

ax.set_xlabel('Mean Temperature (°C)', fontsize=11)
ax.set_ylabel('Std Dev Temperature (°C)', fontsize=11)
ax.set_title('Temperature Mean vs Variability by Country', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
**Next:** [05_feature_importance.ipynb](05_feature_importance.ipynb)